In [1]:
import sys
sys.path.append('/work/lpdi/users/ymiao/code/newcode_0923/atomsurf/')
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf

with initialize(config_path="./conf/"):
    cfg = compose(config_name="config.yaml")
cfg.data_dir='/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/'
cfg.loader.num_workers = 20
cfg.loader.batch_size = 1
cfg.loader.pin_memory =True
cfg.cfg_graph.use_esm=True
OmegaConf.register_new_resolver("eval", eval)
OmegaConf.resolve(cfg)
import torch
torch.multiprocessing.set_sharing_strategy('file_system')
torch.set_num_threads(1)
import hydra
import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from atomsurf.tasks.pip_site.pl_model import PIPsiteModule
from data_loader import PIPsiteDataModule
import warnings
warnings.filterwarnings("ignore")
cfg.min_batch_size=1

/tmp/ipykernel_480157/1559958150.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="./conf/"):


In [3]:
from sklearn.metrics import roc_auc_score

In [2]:

datamodule = PIPsiteDataModule(cfg)

# init model
model = PIPsiteModule(cfg)
loadmodel=model.load_from_checkpoint('/work/lpdi/users/ymiao/code/sbatch/pipsitelog/lightning_logs/version_8_pip_pronet_hmrencoder/checkpoints/last.ckpt')
testloader=datamodule.test_dataloader()

In [13]:
heteroresults=[]
homoresults=[]
count=0
for batchdata in testloader:
    if batchdata.num_graphs>=1 :
        if abs(batchdata.g1_len[0]-batchdata.g2_len[0])>30 :
            output=loadmodel.model(batchdata)
            acc=(torch.sigmoid(output).round().flatten()==torch.cat([batchdata.label_l[0],batchdata.label_r[0]])).sum()/len(output)
            labels_= torch.cat([batchdata.label_l[0],batchdata.label_r[0]]).numpy()
            auroc = roc_auc_score(labels_,torch.sigmoid(output).detach().numpy() )
            # heteroresults.append([output,batchdata])
            heteroresults.append([acc,auroc])
            count+=1
        else:
            output=loadmodel.model(batchdata)
            acc=(torch.sigmoid(output).round().flatten()==torch.cat([batchdata.label_l[0],batchdata.label_r[0]])).sum()/len(output)
            labels_= torch.cat([batchdata.label_l[0],batchdata.label_r[0]]).numpy()
            auroc = roc_auc_score(labels_,torch.sigmoid(output).detach().numpy() )
            # results.append([output,batchdata])
            homoresults.append([acc,auroc])
            count+=1
        print(count)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103



KeyboardInterrupt



In [25]:
heteroresults=np.array(heteroresults)
print(heteroresults[:,0].mean(),heteroresults[:,1].mean())
homoresults=np.array(homoresults)
print(homoresults[:,0].mean(),homoresults[:,1].mean())

0.6015341215663486 0.7209224996519148
0.705922337139354 0.8099189980572081


0.7209224996519148

In [17]:
import numpy as np

In [3]:
results=[]
count=0
for batchdata in testloader:
    if batchdata.num_graphs>=1 :
        if batchdata.g1_len[0]!= batchdata.g2_len[0] and abs(batchdata.g1_len[0]-batchdata.g2_len[0])>50 and batchdata.g1_len[0]>50 and batchdata.g2_len[0]>50 :
            count+=1
            if count< 216:
                continue
            else:
                output=loadmodel.model(batchdata)
                acc=(torch.sigmoid(output).round().flatten()==torch.cat([batchdata.label_l[0],batchdata.label_r[0]])).sum()/len(output)
                results.append([output,batchdata])
                print(count,acc)
                if acc>0.70:
                    print(batchdata.id)

216 tensor(0.5807)
217 tensor(0.6218)
218 tensor(0.5219)
219 tensor(0.5040)
220 tensor(0.5339)
221 tensor(0.4718)
222 tensor(0.4856)
223 tensor(0.5750)
224 tensor(0.5684)
225 tensor(0.6331)
226 tensor(0.6807)
227 tensor(0.5860)
228 tensor(0.6985)
229 tensor(0.5926)
230 tensor(0.5927)
231 tensor(0.5877)
232 tensor(0.5882)
233 tensor(0.5649)
234 tensor(0.5638)
235 tensor(0.5594)
236 tensor(0.5526)
237 tensor(0.4543)
238 tensor(0.5249)
239 tensor(0.5504)
240 tensor(0.5708)
241 tensor(0.5781)
242 tensor(0.6975)
243 tensor(0.7330)
[['2d3t.pdb1.gz_1_B', '2d3t.pdb1.gz_1_D']]
244 tensor(0.5645)
245 tensor(0.6264)
246 tensor(0.4843)
247 tensor(0.8708)
[['3dbh.pdb1.gz_1_C', '3dbh.pdb1.gz_1_D']]
248 tensor(0.8658)
[['3dbh.pdb2.gz_1_A', '3dbh.pdb2.gz_1_B']]
249 tensor(0.8725)
[['3dbh.pdb3.gz_1_G', '3dbh.pdb3.gz_1_H']]
250 tensor(0.8742)
[['3dbh.pdb4.gz_1_E', '3dbh.pdb4.gz_1_F']]
251 tensor(0.8693)
[['3dbl.pdb1.gz_1_G', '3dbl.pdb1.gz_1_H']]
252 tensor(0.8540)
[['3dbl.pdb2.gz_1_C', '3dbl.pdb2.gz_1_D


KeyboardInterrupt



In [13]:
count=0
for data in results:
    if '3dbh.pdb4.gz_1_E' in data[1].id[0]:
        print(count)
        break
    count+=1


34


In [ ]:
results=[]
count=0
for batchdata in testloader:
    if batchdata.num_graphs>=1 :
        if batchdata.g1_len[0]!= batchdata.g2_len[0] and abs(batchdata.g1_len[0]-batchdata.g2_len[0])>50:
            count+=1
            if count< 109:
                continue
            else:
                output=loadmodel.model(batchdata)
                acc=(torch.sigmoid(output).round().flatten()==torch.cat([batchdata.label_l[0],batchdata.label_r[0]])).sum()/len(output)
                results.append([output,batchdata])
            print(count)
                if acc>0.75:
                    print(acc,batchdata.id)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
tensor(0.7889) [['3amb.pdb1.gz_1_A', '3amb.pdb1.gz_1_B']]
31
32
tensor(0.7905) [['1apm.pdb1.gz_1_E', '1apm.pdb1.gz_1_I']]
33
tensor(0.8107) [['1atp.pdb1.gz_1_E', '1atp.pdb1.gz_1_I']]
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
tensor(0.7915) [['4axa.pdb1.gz_1_A', '4axa.pdb1.gz_1_I']]
71
72
tensor(0.7857) [['4b1v.pdb2.gz_1_A', '4b1v.pdb2.gz_1_M']]
73
74
75
tensor(0.7540) [['4b3h.pdb1.gz_1_A', '4b3h.pdb1.gz_1_D']]
76
77
78
79
80
tensor(0.7584) [['4b3i.pdb1.gz_1_B', '4b3i.pdb1.gz_1_C']]
81
82
83
84
85
86
tensor(0.8449) [['4b7x.pdb5.gz_1_I', '4b7x.pdb5.gz_1_J']]
87
tensor(0.7973) [['4b8l.pdb1.gz_1_A', '4b8l.pdb1.gz_1_D']]
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109


[['4dhj.pdb2.gz_1_E', '4dhj.pdb2.gz_1_G']]

In [ ]:
torch.sigmoid(results[0][0])

In [ ]:

from Bio.PDB import PDBParser, PDBIO


# Load the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_left_pred[count]*100)
                # if all_left_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+"_predall.pdb")


parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_right_pred[count]*100)
                # if all_right_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+"_predall.pdb")

tensor(357)

In [12]:
for batchdata in testloader:
    if batchdata.num_graphs>0:
        output=loadmodel.model(batchdata)
        acc=(torch.sigmoid(output).round().flatten()==torch.cat([batchdata.label_l[0],batchdata.label_r[0]])).sum()/len(torch.sigmoid(output).round().flatten())
        print(acc,batchdata.id)
        # break

tensor(0.9500) [['1a05.pdb1.gz_1_A', '1a05.pdb1.gz_1_B']]
tensor(0.7000) [['1a0s.pdb1.gz_1_P', '1a0s.pdb1.gz_1_Q']]
tensor(0.8500) [['1a0s.pdb1.gz_1_P', '1a0s.pdb1.gz_1_R']]
tensor(0.8500) [['1a0s.pdb1.gz_1_Q', '1a0s.pdb1.gz_1_R']]



KeyboardInterrupt



In [14]:
results[34][1].id

[['3dbh.pdb4.gz_1_E', '3dbh.pdb4.gz_1_F']]

In [15]:
i=34
pred=results[i][0]
batchdata=results[i][1]
all_pred= torch.sigmoid(pred)
all_pred_left=all_pred[0:batchdata.g1_len]
all_pred_right=all_pred[batchdata.g1_len:]

all_label=torch.cat([batchdata.label_l[0],batchdata.label_r[0]])

In [16]:
batchdata.idx_left

tensor([  2,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,
         38,  41,  42,  45,  46,  60,  61,  62,  63,  64,  65,  66,  67,  68,
         69,  70,  71,  72,  75,  86,  87,  89,  90,  91, 151, 152, 156, 168,
        169, 170, 171, 204, 319, 320, 321, 322, 323, 324, 325, 327, 328, 331,
        332, 335, 379, 408, 431, 432, 433, 434, 435, 436, 438, 464, 465, 466,
        468, 469, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 483,
        484, 485, 486, 487, 488, 489, 490, 491, 492, 494, 495, 501, 502, 503,
        504, 505, 508, 510, 512, 513, 514, 515, 516, 517, 518, 519,   0,   1,
          3,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,
         30,  31,  32,  33,  34,  35,  36,  37,  39,  40,  43,  44,  47,  48,
         49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  73,  74,  76,
         77,  78,  79,  80,  81,  82,  83,  84,  85,  88,  92,  93,  94,  95,
         96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 1

In [17]:
from Bio.PDB import PDBParser, PDBIO


# Load the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/pdb/"+batchdata.id[0][0]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_pred_left[batchdata.idx_left==count]*100)
                # if all_left_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+"_predall.pdb")
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/pdb/"+batchdata.id[0][1]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_pred_right[batchdata.idx_right==count]*100)
                # if all_right_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+"_predall.pdb")

In [18]:

from Bio.PDB import PDBParser, PDBIO


# Load the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/pdb/"+batchdata.id[0][0]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(batchdata.label_l[0][batchdata.idx_left==count]*100)
                # if all_left_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+"_gd.pdb")


parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/pdb/"+batchdata.id[0][1]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(batchdata.label_r[0][batchdata.idx_right==count]*100)
                # if all_right_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+"_gd.pdb")

In [30]:

from Bio.PDB import PDBParser, PDBIO


# Load the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/pdb/"+batchdata.id[0][0]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_pred[count]*100)
                # if all_left_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+"_predall.pdb")


parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/pdb/"+batchdata.id[0][1]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_pred[count]*100)
                # if all_right_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+"_predall.pdb")

In [31]:

from Bio.PDB import PDBParser, PDBIO


# Load the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/pdb/"+batchdata.id[0][0]+'.pdb')

# Loop through the atoms and modify their B-factors
count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_label[count]*100)
                # if all_left_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][0]+"_gd.pdb")


parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/pdb/"+batchdata.id[0][1]+'.pdb')

# Loop through the atoms and modify their B-factors
# count=0
for model in structure:
    for chain in model:
        for residue in chain:
            # print(residue.full_id)

            for atom in residue:
                atom.set_bfactor(all_label[count]*100)
                # if all_right_pred[count]*100<95:
                #     atom.set_bfactor(0.0)
                # else:
                #     atom.set_bfactor(99.0)
            count+=1

io = PDBIO()
io.set_structure(structure)
io.save("/work/lpdi/users/ymiao/atom2D/data/DIPS-split/data/test/example/"+batchdata.id[0][1]+"_gd.pdb")

[tensor([1., 1., 1., 1., 1., 0., 0., 0., 0., 0.])]

(None, None, None)
